In [1]:
import pandas as pd

df = pd.read_csv("/Users/seongjeyeon/Downloads/aviation-analysis/ntsb_data.csv")
df.shape

(87951, 45)

In [2]:
import os
os.getcwd

<function posix.getcwd()>

In [3]:
df.shape
df.columns.tolist()

['Event Id',
 'Investigation Type',
 'Country',
 'Aircraft Damage',
 'Aircraft Category',
 'Make',
 'Model',
 'Amateur Built',
 'Number Of Engines',
 'Engine Type',
 'Far Description',
 'Schedule',
 'Purpose Of Flight',
 'Total Fatal Injuries',
 'Total Serious Injuries',
 'Total Minor Injuries',
 'Total Uninjured',
 'Weather Condition',
 'Broad Phase Of Flight',
 'Analysis',
 'City',
 'Longitude',
 'Latitude',
 'Address',
 'geometry',
 'Place',
 'Number Of Seats',
 'Type Aircraft',
 'Type Engine',
 'Total Person',
 'Far Description Factorized',
 'Schedule Factorized',
 'Purpose Of Flight Factorized',
 'Make Factorized',
 'Model Factorized',
 'Event Year',
 'Publication Year',
 'Event Month',
 'Publication Month',
 'Event Day',
 'Publication Day',
 'Date Difference',
 'Publication Month Name',
 'Event Month Name',
 'Season']

In [4]:
[c for c in df.columns if 'date' in c.lower() or 'event' in c.lower()]

['Event Id',
 'Event Year',
 'Event Month',
 'Event Day',
 'Date Difference',
 'Event Month Name']

In [5]:
cols = ['Event Year', 'Make', 'Model', 'Broad Phase Of Flight', 'Total Fatal Injuries', 'Analysis']

df_clean = df[cols].copy()
df_clean = df_clean.dropna(subset='Analysis')

df_clean = df[cols].copy()
df_clean = df.dropna(subset=['Analysis'])

df_clean.shape
df_clean.head()

,Event Id,Investigation Type,Country,Aircraft Damage,Aircraft Category,Make,Model,Amateur Built,Number Of Engines,Engine Type,...,Event Year,Publication Year,Event Month,Publication Month,Event Day,Publication Day,Date Difference,Publication Month Name,Event Month Name,Season
0,20001218X45444,Accident,United States,Destroyed,fixed wing single engine,stinson,108-3,No,1,reciprocating,...,1948,2001,10,8,24,24.0,26,August,October,Fall
1,20001218X45447,Accident,United States,Destroyed,weight-shift-control,piper,pa24-180,No,1,reciprocating,...,1962,1996,7,9,19,19.0,34,September,July,Summer
2,20061025X01555,Accident,United States,Destroyed,fixed wing single engine,cessna,172m,No,1,reciprocating,...,1974,2007,8,2,30,30.0,33,February,August,Summer
3,20001218X45448,Accident,United States,Destroyed,weight-shift-control,rockwell,112,No,1,reciprocating,...,1977,2000,6,12,19,19.0,23,December,June,Summer
4,20041105X01764,Accident,United States,Destroyed,fixed wing multi engine,cessna,501,No,2,turbo fan,...,1979,1980,8,4,2,2.0,1,April,August,Summer


In [6]:
cols = ['Event Year', 'Make', 'Model', 'Broad Phase Of Flight',
        'Total Fatal Injuries', 'Analysis']

df_clean = df[cols].copy()
df_clean = df_clean.dropna(subset=['Analysis'])

print(df_clean.shape)
df_clean.head()

(87951, 6)


,Event Year,Make,Model,Broad Phase Of Flight,Total Fatal Injuries,Analysis
0,1948,stinson,108-3,Cruise,2,"\n \n ON OCTOBER 24, 1948, T..."
1,1962,piper,pa24-180,Unknown,4,"\n \n ON JULY 19, 1962, A CO..."
2,1974,cessna,172m,Cruise,3,\n \n The private pilot was ...
3,1977,rockwell,112,Cruise,2,\n \n The aircraft wreckage ...
4,1979,cessna,501,Approach,1,\n \n The Safety Board's ful...


In [7]:
# Filter to only fatal accidents (at least 1 death)
fatal = df_clean[df_clean['Total Fatal Injuries'] > 0]

# top 15 manufacturers by fatal accident count
fatal['Make'].value_counts().head(15)

Make
cessna               5196
piper                3513
beech                1729
bell                  689
robinson              427
mooney                409
boeing                398
bellanca              226
de havilland          169
hughes                157
aerospatiale          140
grumman               137
mcdonnell douglas     135
north american        132
aero commander        129
Name: count, dtype: int64

In [8]:
# Group fatal accidents by year, sort chronologically
fatal_by_year = fatal.groupby('Event Year').size().sort_index()

fatal_by_year

Event Year
1948      1
1962      1
1974      1
1977      1
1979      1
1981      1
1982    678
1983    698
1984    653
1985    593
1986    531
1987    511
1988    539
1989    507
1990    530
1991    513
1992    519
1993    462
1994    492
1995    487
1996    456
1997    431
1998    448
1999    430
2000    477
2001    718
2002    709
2003    719
2004    677
2005    668
2006    636
2007    595
2008    402
2009    375
2010    366
2011    413
2012    410
2013    371
2014    372
2015    389
2016    368
2017    367
2018    401
2019    407
2020    312
2021    303
2022    325
dtype: int64

## Key findings from initial exploration

- Top makes by fatal accident count (Cessna, Piper, Beech) reflect **fleet 
  size and pilot demographics**, not manufacturer quality. General aviation 
  dominates this dataset.
- Pre-1982 data is sparse and unreliable (NTSB digitization era).
- Fatal accidents drop ~40% starting 2008. Likely causes: post-financial-crisis 
  decline in GA flight hours, glass cockpit / ADS-B adoption, aging pilot 
  population.
- **Caveat:** Without flight-hour denominators, all of this is suggestive, 
  not conclusive.

In [9]:
# Drop pre-1982 (sparse / unreliable digitization)
df_clean = df_clean[df_clean['Event Year'] >= 1982]

# Standardize manufacturer names — they're inconsistent
df_clean['Make'] = df_clean['Make'].str.upper().str.strip()

# Confirm the cleanup
print(df_clean.shape)
df_clean['Make'].value_counts().head(10)

(87944, 6)


Make
CESSNA      26836
PIPER       14743
BEECH        5332
BELL         2706
BOEING       2652
MOONEY       1322
ROBINSON     1223
GRUMMAN      1158
BELLANCA     1033
HUGHES        931
Name: count, dtype: int64

In [10]:
fatal = df_clean[df_clean['Total Fatal Injuries'] > 0] 
print(fatal.shape)

(20258, 6)


## Today I learned

That Cessna leading the fatal-accident list isn't a quality signal — it's a 
fleet-size and pilot-demographics signal. Without flight-hour denominators, 
raw counts can mislead. Always ask "per what?" before drawing a conclusion.